# TR-WISARD — Experimentos em todos os datasets
Roda **TrWisard1**  e **TrWisard2**  com os parâmetros ótimos de cada dataset
e salva métricas, gráficos e vídeo em `data/{dataset}/experimentos/experimento_N/`
(auto-incrementado a cada execução).

In [1]:
import sys, time, json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.tr_wisard import TRWisard
from src.dataset import load_dataset, preload_frames
from src.metrics import save_comparison, create_experiment_folder

summary = {}

/Users/isabellagoncalves/personal_project/TR-WISARD/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Parâmetros por dataset
Parâmetros ótimos tunados, copiados manualmente para os dicionários abaixo — edite aqui
diretamente para testar outros valores (nada é lido de arquivo em tempo de execução).

**TrWisard1**  — valores tunados por `tune_datasets.py` / `tune_faceocc.py` / `tune_tiger1.py`.

**TrWisard2**  — Tabela 4.6.

In [ ]:
_CLUS_FIXED = {
    'CLUSWISARD_THRESHOLD':           1,
    'CLUSWISARD_BLEACHING_ACTIVATED': True,
    'CLUSWISARD_ACTIVATION_DEGREE':   True,
    'CLUSWISARD_RETURN_CONFIDENCE':   True,
    'CLUSWISARD_CLASSES_DEGREES':     True,
    'REMOVE_BACKGROUND': False,
    'BACKGROUND_ALPHA':  0,
    'PASS_SCORE':         0.90,
    'SEED':               21,
}

# TrWisard1  — parâmetros tunados por dataset
# Addr | Min.Score | Disc.Limit | Anchor | Scale | Step
PARAMS_TRWISARD1 = {
    'dollar':   {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 7, 'CLUSWISARD_MIN_SCORE': 0.20, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  5, 'ANCHOR_SCORE': 0.05, 'MAX_SEARCH_WINDOW_SCALE': 0.2, 'STEP_SIZE': 3},
    'david':    {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 7, 'CLUSWISARD_MIN_SCORE': 0.15, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  5, 'ANCHOR_SCORE': 0.05, 'MAX_SEARCH_WINDOW_SCALE': 0.2, 'STEP_SIZE': 5},
    'faceocc':  {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 5, 'CLUSWISARD_MIN_SCORE': 0.15, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  3, 'ANCHOR_SCORE': 0.05, 'MAX_SEARCH_WINDOW_SCALE': 0.2, 'STEP_SIZE': 2},
    'faceocc2': {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 9, 'CLUSWISARD_MIN_SCORE': 0.40, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  3, 'ANCHOR_SCORE': 0.05, 'MAX_SEARCH_WINDOW_SCALE': 0.2, 'STEP_SIZE': 5},
    'sylv':     {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 5, 'CLUSWISARD_MIN_SCORE': 0.30, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  3, 'ANCHOR_SCORE': 0.30, 'MAX_SEARCH_WINDOW_SCALE': 0.2, 'STEP_SIZE': 3},
    'tiger1':   {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 5, 'CLUSWISARD_MIN_SCORE': 0.65, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  3, 'ANCHOR_SCORE': 0.30, 'MAX_SEARCH_WINDOW_SCALE': 0.5, 'STEP_SIZE': 5},
    'tiger2':   {**_CLUS_FIXED, 'CLUSWISARD_ADDRESS_SIZE': 9, 'CLUSWISARD_MIN_SCORE': 0.50, 'CLUSWISARD_DISCRIMINATOR_LIMIT':  3, 'ANCHOR_SCORE': 0.15, 'MAX_SEARCH_WINDOW_SCALE': 0.5, 'STEP_SIZE': 3},
}

_WISD_FIXED = {'SEED': 21}

# TrWisard2  — parâmetros tunados por dataset
# Addr | Lim.Ret | Lim.ND | Queue | Max.Ret | Radius | Step | BG Rem | α
PARAMS_TRWISARD2 = {
    'dollar':   {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.80, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': False, 'BACKGROUND_ALPHA': 0.10},
    'david':    {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.80, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 1, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 1.00},
    'faceocc':  {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.80, 'LIMIAR_NOVO_DISC': 0.20, 'QUEUE_MAX_SIZE':  5, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 1.00},
    'faceocc2': {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 5, 'LIMIAR_RETREINO': 0.50, 'LIMIAR_NOVO_DISC': 0.40, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 10, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 1.00},
    'sylv':     {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.50, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 25, 'MAX_RETRAINS': 1, 'SEARCH_RADIUS':  5, 'STEP_SIZE': 5, 'REMOVE_BACKGROUND': True,  'BACKGROUND_ALPHA': 0.10},
    'tiger1':   {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 3, 'LIMIAR_RETREINO': 0.50, 'LIMIAR_NOVO_DISC': 0.60, 'QUEUE_MAX_SIZE': 10, 'MAX_RETRAINS': 1, 'SEARCH_RADIUS': 30, 'STEP_SIZE': 3, 'REMOVE_BACKGROUND': False, 'BACKGROUND_ALPHA': 0},
    'tiger2':   {**_WISD_FIXED, 'WISARD_ADDRESS_SIZE': 5, 'LIMIAR_RETREINO': 0.65, 'LIMIAR_NOVO_DISC': 0.40, 'QUEUE_MAX_SIZE':  8, 'MAX_RETRAINS': 3, 'SEARCH_RADIUS': 20, 'STEP_SIZE': 3, 'REMOVE_BACKGROUND': False, 'BACKGROUND_ALPHA': 0},
}

## Execução por dataset
Cada célula abaixo é independente — execute uma de cada vez ou todas em sequência.

In [3]:
# ── dollar ────────────────────────────────────────────────────────────────────
dataset = 'dollar'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'Tr-WiSARD1', elapsed_1,
    preds_2, 'Tr-WiSARD2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== dollar ===
Pré-carregando 327 frames...
  Done em 0.7s
Pasta: experimento_1


TrWisard2: 100%|██████████| 326/326 [00:12<00:00, 26.20it/s]


  Tr-WiSARD1    CLE=2.69 px  Jaccard=91.34%  FPS=17.8
  Tr-WiSARD2    CLE=2.58 px  Jaccard=92.09%  FPS=26.3


In [5]:
# ── david ─────────────────────────────────────────────────────────────────────
dataset = 'david'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== david ===
Pré-carregando 462 frames...
  Done em 0.6s
Pasta: experimento_2


TrWisard2: 100%|██████████| 461/461 [00:57<00:00,  7.97it/s]


  TrWisard1     CLE=7.91 px  Jaccard=80.73%  FPS=39.3
  TrWisard2     CLE=6.21 px  Jaccard=83.78%  FPS=8.0


In [ ]:
# ── faceocc ───────────────────────────────────────────────────────────────────
dataset = 'faceocc'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== faceocc ===
Pré-carregando 888 frames...


In [7]:
# ── faceocc2 ──────────────────────────────────────────────────────────────────
dataset = 'faceocc2'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== faceocc2 ===
Pré-carregando 816 frames...
  Done em 1.5s
Pasta: experimento_1


TrWisard2:  95%|█████████▌| 778/815 [00:48<00:02, 16.19it/s]


KeyboardInterrupt: 

In [9]:
# ── sylv ──────────────────────────────────────────────────────────────────────
dataset = 'sylv'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== sylv ===
Pré-carregando 1345 frames...
  Done em 1.0s
Pasta: experimento_2


TrWisard1:  12%|█▏        | 155/1344 [00:00<00:06, 187.04it/s]


KeyboardInterrupt: 

In [3]:
# ── tiger1 ────────────────────────────────────────────────────────────────────
dataset = 'tiger1'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== tiger1 ===
Pré-carregando 354 frames...
  Done em 0.5s
Pasta: experimento_1


TrWisard2: 100%|██████████| 353/353 [00:12<00:00, 28.98it/s]


  TrWisard1     CLE=9.27 px  Jaccard=66.48%  FPS=110.9
  TrWisard2     CLE=9.10 px  Jaccard=67.38%  FPS=29.1


In [4]:
# ── tiger2 ────────────────────────────────────────────────────────────────────
dataset = 'tiger2'
print(f'=== {dataset} ===')
image_paths, ground_truths = load_dataset(dataset)
frames = preload_frames(image_paths)

exp_path = create_experiment_folder(PROJECT_ROOT / 'data' / dataset / 'experimentos')
print(f'Pasta: {exp_path.name}')

params_1 = PARAMS_TRWISARD1[dataset]
params_2 = PARAMS_TRWISARD2[dataset]
with open(exp_path / 'params_trwisard1.json', 'w') as f:
    json.dump(params_1, f, indent=4)
with open(exp_path / 'params_trwisard2.json', 'w') as f:
    json.dump(params_2, f, indent=4)

t0 = time.time()
preds_1 = TRWisard('tr_wisard1', frames, ground_truths, params=params_1).run()
elapsed_1 = time.time() - t0

t0 = time.time()
preds_2 = TRWisard('tr_wisard2', frames, ground_truths, params=params_2).run()
elapsed_2 = time.time() - t0

result = save_comparison(
    dataset, exp_path,
    preds_1, 'TrWisard1', elapsed_1,
    preds_2, 'TrWisard2', elapsed_2,
    ground_truths, frames=frames,
)
summary[dataset] = result
for name, m in result.items():
    print(f'  {name:12s}  CLE={m["mean_cle"]:.2f} px  Jaccard={m["mean_jaccard"]:.2f}%  FPS={m["fps"]:.1f}')

=== tiger2 ===
Pré-carregando 365 frames...
  Done em 0.5s
Pasta: experimento_1


TrWisard2: 100%|██████████| 364/364 [00:19<00:00, 18.89it/s]


  TrWisard1     CLE=6.43 px  Jaccard=68.50%  FPS=71.4
  TrWisard2     CLE=11.11 px  Jaccard=54.34%  FPS=18.9


## Tabela resumo
Execute após rodar os datasets desejados acima.

In [ ]:
rows = []
for ds, r in summary.items():
    for mode, m in r.items():
        rows.append({
            'Dataset':           ds,
            'Modo':              mode,
            'CLE medio (px)':    round(m['mean_cle'], 2),
            'Jaccard medio (%)': round(m['mean_jaccard'], 2),
            'FPS':               round(m['fps'], 1),
        })

df = pd.DataFrame(rows)
df